# 05 — Expanded Autoencoder Training + CD8 T Cell Holdout

**Changes from previous experiment (04):**
1. Train scGen on ALL ~106K cells (not just ~12K matched) to reduce overfitting
2. Hold out CD8+ T cells (`CL:0000625`) instead of broad T cell (`CL:0000084`)
   - Model sees: T cells, thymocytes, CD4+ T cells
   - Model does NOT see: CD8+ T cells

**Outputs:**
- `ae_training_expanded.h5ad` — all ~106K cells x 1000 HVGs (for scGen)
- `cd8_holdout.h5ad` — ~12K matched cells x 1000 HVGs (for CellOT)

## Section 1 — Imports and Data Loading

In [1]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

import importlib
if "speciesot_helpers" in sys.modules:
    importlib.reload(sys.modules["speciesot_helpers"])

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from speciesot_helpers import (
    top_n_organisms_from_species,
    match_cells_by_celltype_tissue,
    align_adatas_biomart_one2one,
    strip_ensembl_gene_id,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Imports OK")

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
human_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/'
mouse_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/'

human_adatas = top_n_organisms_from_species(human_dir, -1, 'human')
mouse_adatas = top_n_organisms_from_species(mouse_dir, -1, 'mouse')

human_combined = ad.concat(human_adatas, join="outer")
mouse_combined = ad.concat(mouse_adatas, join="outer")

print(f"Human combined: {human_combined.shape}")
print(f"Mouse combined: {mouse_combined.shape}")
print(f"Total cells:    {human_combined.n_obs + mouse_combined.n_obs}")

Human combined: (58852, 61759)
Mouse combined: (47802, 18024)
Total cells:    106654


## Section 2 — BioMart Gene Alignment on FULL Datasets

In [3]:
mouse_is_ensembl = any(str(g).upper().startswith('ENSMUSG') for g in mouse_combined.var_names[:20])
human_is_ensembl = any(str(g).upper().startswith('ENSG') for g in human_combined.var_names[:20])
print(f"Mouse uses Ensembl IDs: {mouse_is_ensembl}")
print(f"Human uses Ensembl IDs: {human_is_ensembl}")

assert mouse_is_ensembl and human_is_ensembl, \
    "Both datasets must use Ensembl IDs for BioMart alignment"

print("\nAligning genes via BioMart one-to-one orthologs (this queries Ensembl, may take a minute) ...")
mouse_all_aligned, human_all_aligned, ortholog_table = align_adatas_biomart_one2one(
    mouse_combined, human_combined
)

print(f"\nOrtholog pairs (overlap with both datasets): {len(ortholog_table)}")
print(f"Mouse aligned: {mouse_all_aligned.shape}")
print(f"Human aligned: {human_all_aligned.shape}")
print(f"Gene names match: {list(mouse_all_aligned.var_names[:3]) == list(human_all_aligned.var_names[:3])}")

Mouse uses Ensembl IDs: True
Human uses Ensembl IDs: True

Aligning genes via BioMart one-to-one orthologs (this queries Ensembl, may take a minute) ...

Ortholog pairs (overlap with both datasets): 14451
Mouse aligned: (47802, 14451)
Human aligned: (58852, 14451)
Gene names match: True


## Section 3 — HVG Selection on ALL Cells

In [4]:
N_HVG = 1000

mouse_all_aligned.obs['condition'] = 'mouse'
human_all_aligned.obs['condition'] = 'human'

all_cells = ad.concat([mouse_all_aligned, human_all_aligned], join='inner')
print(f"Concatenated all cells: {all_cells.shape}")

sc.pp.highly_variable_genes(all_cells, n_top_genes=N_HVG, flavor='seurat')
hvg_genes = all_cells.var_names[all_cells.var.highly_variable].tolist()
print(f"Selected {len(hvg_genes)} HVGs from full pool")

mouse_all_hvg = mouse_all_aligned[:, hvg_genes].copy()
human_all_hvg = human_all_aligned[:, hvg_genes].copy()

print(f"\nMouse all HVG: {mouse_all_hvg.shape}")
print(f"Human all HVG: {human_all_hvg.shape}")
print(f"Total for AE training: {mouse_all_hvg.n_obs + human_all_hvg.n_obs}")

Concatenated all cells: (106654, 14451)
Selected 1000 HVGs from full pool

Mouse all HVG: (47802, 1000)
Human all HVG: (58852, 1000)
Total for AE training: 106654


## Section 4 — Extract Matched Subset for CellOT

In [5]:
mouse_matched_hvg, human_matched_hvg = match_cells_by_celltype_tissue(
    mouse_all_hvg, human_all_hvg
)

print(f"Matched mouse: {mouse_matched_hvg.shape}")
print(f"Matched human: {human_matched_hvg.shape}")
print(f"Gene names match: {list(mouse_matched_hvg.var_names) == list(human_matched_hvg.var_names)}")

Matched mouse: (6418, 1000)
Matched human: (6418, 1000)
Gene names match: True


## Section 5 — Inspect CD8 Holdout Feasibility

In [6]:
HOLDOUT_CELLTYPE = "CL:0000625"  # CD8-positive, alpha-beta T cell

T_CELL_FAMILY = {
    "CL:0000084": "T cell",
    "CL:0000893": "thymocyte",
    "CL:0000624": "CD4-positive, alpha-beta T cell",
    "CL:0000625": "CD8-positive, alpha-beta T cell",
}

m_obs = mouse_matched_hvg.obs.copy()
h_obs = human_matched_hvg.obs.copy()
m_obs['species'] = 'mouse'
h_obs['species'] = 'human'
matched_obs = pd.concat([m_obs, h_obs], axis=0)

print("T-cell family in matched dataset:")
print("=" * 70)
for cid, name in T_CELL_FAMILY.items():
    mask = matched_obs['cell_type_ontology_term_id'].astype(str) == cid
    n_total = mask.sum()
    n_mouse = (mask & (matched_obs['species'] == 'mouse')).sum()
    n_human = (mask & (matched_obs['species'] == 'human')).sum()
    role = "HOLDOUT" if cid == HOLDOUT_CELLTYPE else "in training"
    print(f"  {cid} ({name}): {n_total} total ({n_mouse} mouse, {n_human} human) — {role}")

holdout_count = (matched_obs['cell_type_ontology_term_id'].astype(str) == HOLDOUT_CELLTYPE).sum()
print(f"\nHoldout CD8+ T cells: {holdout_count} total")
print(f"After toggle_ood 50/50 split: ~{holdout_count // 2} OOD, ~{holdout_count // 2} ignore")

print(f"\nAll cell types in matched data ({matched_obs['cell_type_ontology_term_id'].nunique()} unique):")
ct_counts = (
    matched_obs.groupby(['cell_type_ontology_term_id', 'species'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
ct_counts['total'] = ct_counts.get('mouse', 0) + ct_counts.get('human', 0)
ct_counts = ct_counts.sort_values('total', ascending=False).reset_index(drop=True)
display(ct_counts)

T-cell family in matched dataset:
  CL:0000084 (T cell): 204 total (102 mouse, 102 human) — in training
  CL:0000893 (thymocyte): 910 total (455 mouse, 455 human) — in training
  CL:0000624 (CD4-positive, alpha-beta T cell): 190 total (95 mouse, 95 human) — in training
  CL:0000625 (CD8-positive, alpha-beta T cell): 390 total (195 mouse, 195 human) — HOLDOUT

Holdout CD8+ T cells: 390 total
After toggle_ood 50/50 split: ~195 OOD, ~195 ignore

All cell types in matched data (30 unique):


/tmp/ipykernel_3895670/2306212661.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  matched_obs.groupby(['cell_type_ontology_term_id', 'species'])


species,cell_type_ontology_term_id,human,mouse,total
0,CL:0008001,792,792,1584
1,CL:0000037,721,721,1442
2,CL:0002393,504,504,1008
3,CL:0000623,456,456,912
4,CL:0000893,455,455,910
5,CL:0000875,426,426,852
6,CL:1000320,380,380,760
7,CL:0002063,292,292,584
8,CL:0002543,287,287,574
9,CL:0002548,242,242,484


## Section 6 — Export h5ad Files

In [7]:
BASE_DIR = '/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT'
DATASET_DIR = os.path.join(BASE_DIR, 'cellot/cellot_gpu/datasets/speciesot-human-mouse')
os.makedirs(DATASET_DIR, exist_ok=True)

from scipy import sparse as sp_sparse

keep_obs = ['condition', 'species', 'cell_type_ontology_term_id', 'cell_type',
            'tissue_ontology_term_id', 'tissue', 'donor_id']

def clean_adata(adata):
    """Strip layers/obsm/uns and densify X for compatibility with older anndata in CellOT env."""
    obs_cols = [c for c in keep_obs if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=adata.obs[obs_cols].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )

# --- File 1: Expanded AE training data (CD8+ EXCLUDED for strict OOD) ---
mouse_all_hvg.obs['condition'] = 'mouse'
human_all_hvg.obs['condition'] = 'human'
ae_data = ad.concat([mouse_all_hvg, human_all_hvg], join='inner')
n_before = ae_data.n_obs
ae_data = ae_data[
    ae_data.obs['cell_type_ontology_term_id'].astype(str) != HOLDOUT_CELLTYPE
].copy()
ae_data = clean_adata(ae_data)
print(f"AE training: {n_before} total -> {ae_data.n_obs} after excluding CD8+ ({HOLDOUT_CELLTYPE})")

ae_path = os.path.join(DATASET_DIR, 'ae_training_expanded.h5ad')
ae_data.write_h5ad(ae_path)
print(f"Saved AE training data: {ae_path}")
print(f"  Shape: {ae_data.shape}")
print(f"  Conditions: {dict(ae_data.obs['condition'].value_counts())}")

# --- File 2: Matched CellOT data (unswapped: condition=species) ---
mouse_matched_hvg.obs['condition'] = 'mouse'
human_matched_hvg.obs['condition'] = 'human'
cellot_data = ad.concat([mouse_matched_hvg, human_matched_hvg], join='inner')
cellot_data = clean_adata(cellot_data)

cellot_path = os.path.join(DATASET_DIR, 'cd8_holdout.h5ad')
cellot_data.write_h5ad(cellot_path)
print(f"\nSaved CellOT unswapped data: {cellot_path}")
print(f"  Shape: {cellot_data.shape}")
print(f"  Conditions: {dict(cellot_data.obs['condition'].value_counts())}")
print(f"  Cell types: {cellot_data.obs['cell_type_ontology_term_id'].nunique()} unique")
n_cd8 = (cellot_data.obs['cell_type_ontology_term_id'] == HOLDOUT_CELLTYPE).sum()
print(f"  CD8+ T cells (CL:0000625): {n_cd8}")

# --- File 3: Matched CellOT data (swapped: condition=cell_type_status) ---
# condition = 'cd8' or 'non_cd8'; species column preserved for holdout
swapped = ad.concat([mouse_matched_hvg, human_matched_hvg], join='inner')
swapped.obs['species'] = swapped.obs['condition'].values  # preserve species
swapped.obs['condition'] = np.where(
    swapped.obs['cell_type_ontology_term_id'].astype(str) == HOLDOUT_CELLTYPE,
    'cd8', 'non_cd8'
)
swapped = clean_adata(swapped)

swapped_path = os.path.join(DATASET_DIR, 'cd8_holdout_swapped.h5ad')
swapped.write_h5ad(swapped_path)
print(f"\nSaved CellOT swapped data: {swapped_path}")
print(f"  Shape: {swapped.shape}")
print(f"  condition values: {dict(swapped.obs['condition'].value_counts())}")
print(f"  species values:   {dict(swapped.obs['species'].value_counts())}")

AE training: 106654 total -> 104656 after excluding CD8+ (CL:0000625)
Saved AE training data: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_expanded.h5ad
  Shape: (104656, 1000)
  Conditions: {'human': np.int64(57854), 'mouse': np.int64(46802)}

Saved CellOT unswapped data: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout.h5ad
  Shape: (12836, 1000)
  Conditions: {'human': np.int64(6418), 'mouse': np.int64(6418)}
  Cell types: 30 unique
  CD8+ T cells (CL:0000625): 390

Saved CellOT swapped data: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_swapped.h5ad
  Shape: (12836, 1000)
  condition values: {'non_cd8': np.int64(12446), 'cd8': np.int64(390)}
  species values:   {'human': np.int64(6418), 'mouse': np.int64(6418)}


## Section 7 — Summary and Suggested Config Updates

In [8]:
n_ae_cells = ae_data.n_obs
n_genes = ae_data.n_vars

params_512 = n_genes * 512 + 512 * 512 + 512 * 50  # ~800K for [512,512]
params_256 = n_genes * 256 + 256 * 256 + 256 * 50   # ~330K for [256,256]

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"\nAE training cells:   {n_ae_cells:,}")
print(f"CellOT matched cells: {cellot_data.n_obs:,} ({cellot_data.n_obs // 2:,} per species)")
print(f"Gene dimensions:      {n_genes}")
print(f"CD8+ holdout cells:   {n_cd8}")
print(f"\nParameter counts (approx):")
print(f"  hidden=[512,512]: ~{params_512:,} params  →  {n_ae_cells / params_512:.2f} cells/param")
print(f"  hidden=[256,256]: ~{params_256:,} params  →  {n_ae_cells / params_256:.2f} cells/param")
print(f"\nPrevious experiment (04):")
print(f"  AE training cells: ~10,000")
print(f"  hidden=[512,512]: ~800K params → 0.01 cells/param (high overfitting risk)")

print("\n" + "=" * 70)
print("SUGGESTED YAML CONFIGS")
print("=" * 70)

print("""
--- scGen (ae_training_expanded.h5ad) ---

  configs/tasks/speciesot-cd8-ood-ae.yaml:

    data:
      type: cell
      source: mouse
      target: human
      condition: condition
      path: datasets/speciesot-human-mouse/ae_training_expanded.h5ad

    datasplit:
      groupby: condition
      name: train_test
      test_size: 0.2
      random_state: 0

  Note: scGen trains on ALL cells (no holdout here).
  The holdout is only applied during CellOT training.

--- CellOT (cd8_holdout.h5ad) ---

  configs/tasks/speciesot-cd8-ood-cellot.yaml:

    data:
      type: cell
      source: human
      target: mouse
      condition: condition
      path: datasets/speciesot-human-mouse/cd8_holdout.h5ad
      ae_emb:
        path: ./results/<scgen_results_dir>/

    datasplit:
      holdout: "CL:0000625"
      key: cell_type_ontology_term_id
      groupby: condition
      name: toggle_ood
      mode: ood
      test_size: 0.2
      random_state: 0

--- Optional: reduce scGen architecture ---

  model:
    name: scgen
    beta: 0.0
    dropout: 0.1
    hidden_units: [256, 256]
    latent_dim: 50
""")

print("Done. Run scGen first on ae_training_expanded.h5ad, then CellOT on cd8_holdout.h5ad.")

SUMMARY

AE training cells:   104,656
CellOT matched cells: 12,836 (6,418 per species)
Gene dimensions:      1000
CD8+ holdout cells:   390

Parameter counts (approx):
  hidden=[512,512]: ~799,744 params  →  0.13 cells/param
  hidden=[256,256]: ~334,336 params  →  0.31 cells/param

Previous experiment (04):
  AE training cells: ~10,000
  hidden=[512,512]: ~800K params → 0.01 cells/param (high overfitting risk)

SUGGESTED YAML CONFIGS

--- scGen (ae_training_expanded.h5ad) ---

  configs/tasks/speciesot-cd8-ood-ae.yaml:

    data:
      type: cell
      source: mouse
      target: human
      condition: condition
      path: datasets/speciesot-human-mouse/ae_training_expanded.h5ad

    datasplit:
      groupby: condition
      name: train_test
      test_size: 0.2
      random_state: 0

  Note: scGen trains on ALL cells (no holdout here).
  The holdout is only applied during CellOT training.

--- CellOT (cd8_holdout.h5ad) ---

  configs/tasks/speciesot-cd8-ood-cellot.yaml:

    data:
  